In [1]:
import os

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken
from autogen_agentchat.base import TaskResult

from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console


from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

In [2]:
import dotenv
dotenv.load_dotenv()
client = AzureAIChatCompletionClient(
    model=os.getenv("MODEL_FREE_8B"),
    endpoint=os.getenv("API_URL"),
    credential=AzureKeyCredential(os.getenv("API_KEY")),
    model_info={
        "json_output": True,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

/home/dsa/workspace/ai-agents-for-beginners/venv/lib/python3.12/site-packages/autogen_ext/models/azure/_azure_ai_client.py:307: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(config["model_info"])


In [3]:
frontdesk_agent = AssistantAgent(
    "planner_agent",
    model_client=client,
    description="一个可以规划旅行的有用助手。",
    system_message="""
    您是一位拥有十年经验的前台旅行代理，因需要处理许多客户而以简洁著称。
    目标是为旅行者提供最好的活动和地点。
    每次回复只提供一个推荐。
    您专注于当前的目标。
    不要浪费时间在闲聊上。
    在完善想法时考虑建议。""",
)

concierge_agent = AssistantAgent(
    "concierge_agent",
    model_client=client,
    description="一个可以建议当地活动或参观地点的本地助手。",
    system_message="""
    您是一位酒店礼宾员，对为旅行者提供最本地化和最真实的体验有自己的看法。
    目标是确定前台旅行代理是否为旅行者推荐了最好的非旅游体验。
    如果是，请回复 'APPROVE'
    如果不是，请提供关于如何完善推荐的见解，不要使用具体示例。
    """,
)

In [4]:
termination = TextMentionTermination("APPROVE")
team = RoundRobinGroupChat(
    [frontdesk_agent, concierge_agent], termination_condition=termination
)

async for message in team.run_stream(task="我想计划一次去巴黎的旅行。"):
    if isinstance(message, TaskResult):
        print("停止原因:", message.stop_reason)
    else:
        print(message)

id='76707102-576f-460c-a58c-2d9facca34ae' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 2, 8, 16, 59, 11, 227576, tzinfo=datetime.timezone.utc) content='我想计划一次去巴黎的旅行。' type='TextMessage'
id='0b8d9e36-f395-4861-aaf8-9c997639ba66' source='planner_agent' models_usage=RequestUsage(prompt_tokens=98, completion_tokens=17) metadata={} created_at=datetime.datetime(2026, 2, 8, 16, 59, 12, 216187, tzinfo=datetime.timezone.utc) content='您想首选哪个区域？左岸文艺范还是右岸经典名胜？' type='TextMessage'
id='9cc05e77-0b7c-485d-a0e4-8fdefc419fd5' source='concierge_agent' models_usage=RequestUsage(prompt_tokens=120, completion_tokens=156) metadata={} created_at=datetime.datetime(2026, 2, 8, 16, 59, 19, 188912, tzinfo=datetime.timezone.utc) content='APPROVE\n\n在巴黎，左岸与右岸各自代表了不同的体验风格，选择取决于您对旅行的偏好。左岸以文艺气息、咖啡馆文化、艺术氛围和相对宁静的社区著称，适合喜欢探索、追求独特体验和深度文化的旅行者。右岸则是经典名胜区，汇聚了卢浮宫、埃菲尔铁塔、巴黎圣母院等标志性建筑，适合希望沉浸在巴黎历史与地标景观中的游客。\n\n作为礼宾员，我理解并尊重这两种风格的吸引力。如果旅行者明确表达了他们对体验类型的偏好，例如更倾向于文化探索而非观光打卡，我会根据其兴趣推荐最契合的区域。因此，推荐是基于旅行者个